In [16]:
# Imports
import pandas as pd
import torch
import torch.nn as nn # just get that all up in here. 
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


from torch import optim

from torch.optim.lr_scheduler import StepLR
from imblearn.over_sampling import SMOTE
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset, DataLoader

# for toy model, use a SUBSET so we aren't tearing our hair out trying to run this. 
from torch.utils.data import Subset
# same thing that we did for the MLP
from sklearn.metrics import (roc_auc_score, confusion_matrix, 
                             classification_report, fbeta_score)
import numpy as np

In [17]:
class DenseLayer(nn.Module):
    """single dense layer: BN → ReLU → Conv"""
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1, bias=False),
            nn.Dropout(p=0.2)
        )

    def forward(self, x):
        new_features = self.layer(x)
        # concatenate input with output — this is the dense connection
        return torch.cat([x, new_features], dim=1)


class DenseBlock(nn.Module):
    """several dense layers chained together"""
    def __init__(self, in_channels, num_layers, growth_rate):
        super().__init__()
        self.layers = nn.ModuleList()
        current_channels = in_channels

        for _ in range(num_layers):
            self.layers.append(DenseLayer(current_channels, growth_rate))
            current_channels += growth_rate  # channels grow by growth_rate each layer

        self.out_channels = current_channels

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class TransitionLayer(nn.Module):
    """compresses channels and halves spatial size between dense blocks"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        return self.layer(x)


class MiniDenseNet(nn.Module):
    """
    Scaled down DenseNet — same architecture as DenseNet-121
    but fewer layers per block so it's trainable on your hardware.
    
    DenseNet-121 has blocks of [6, 12, 24, 16] layers
    This has blocks of         [4,  8, 12,  8] layers — same shape, fewer params
    """
    def __init__(self, growth_rate=32, num_classes=1):
        super().__init__()

        # initial convolution — same as DenseNet-121
        self.initial = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        # output: (batch, 64, 56, 56) for 224px input

        # dense blocks + transitions
        # block 1
        self.block1 = DenseBlock(64, num_layers=4, growth_rate=growth_rate)
        ch = self.block1.out_channels  # 64 + 4*32 = 192
        self.trans1 = TransitionLayer(ch, ch // 2)
        ch = ch // 2  # 96

        # block 2
        self.block2 = DenseBlock(ch, num_layers=8, growth_rate=growth_rate)
        ch = self.block2.out_channels  # 96 + 8*32 = 352
        self.trans2 = TransitionLayer(ch, ch // 2)
        ch = ch // 2  # 176

        # block 3
        self.block3 = DenseBlock(ch, num_layers=12, growth_rate=growth_rate)
        ch = self.block3.out_channels  # 176 + 12*32 = 560
        self.trans3 = TransitionLayer(ch, ch // 2)
        ch = ch // 2  # 280

        # block 4
        self.block4 = DenseBlock(ch, num_layers=8, growth_rate=growth_rate)
        ch = self.block4.out_channels  # 280 + 8*32 = 536

        # classifier
        self.classifier = nn.Sequential(
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(ch, num_classes)
            # no sigmoid — BCEWithLogitsLoss handles it
        )

        self.out_channels = ch

    def forward(self, x):
        x = self.initial(x)
        x = self.trans1(self.block1(x))
        x = self.trans2(self.block2(x))
        x = self.trans3(self.block3(x))
        x = self.block4(x)
        x = self.classifier(x)
        return x

In [18]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [19]:
model = MiniDenseNet().to(device)
dummy = torch.randn(4, 1, 224, 224).to(device)
out   = model(dummy)
print(f"Output shape: {out.shape}")  # (4, 1)

# count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")  # expect ~2-4M vs DenseNet-121's 7M

Output shape: torch.Size([4, 1])
Parameters: 2,991,145
